In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS

In [8]:
load_dotenv()

True

In [6]:
doc1 = Document(page_content="LangChain makes it easy to work with LLMs.")
doc2 = Document(page_content="LangChain is used to build LLM based applications.")
doc3 = Document(page_content="Chroma is used to store and search document embeddings.")
doc4 = Document(page_content="Embeddings are vector representations of text.")
doc5 = Document(page_content="MMR helps you get diverse results when doing similarity search.")
doc6 = Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more.")

docs = [doc1, doc2, doc3, doc4, doc5, doc6]

In [9]:
vec_store = FAISS.from_documents(
    documents = docs,
    embedding = OpenAIEmbeddings(),
    ids = ['1', '2', '3', '4', '5', '6'])

In [14]:
# Initializing retriever

retriever = vec_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = {'k': 2, 'lambda_mult': 0.5}
)

# If lambda mult is set to 1, it will act same as 'similarity',
# if set to 0, it will give totally diversed results (Default is 0.5)
# So we should keep it between 0 and 1

In [15]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000022A67E505D0>, search_type='mmr', search_kwargs={'k': 2, 'lambda_mult': 0.5})

In [16]:
result = retrieval.invoke('What is langchain?')

result

# See, it didn't return both documents from docs which are highly internally similar

# LangChain makes it easy to work with LLMs
# LangChain is used to build LLM based applications.

# If we had used simple search method, it would have return these both decuments.

[Document(id='2', metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(id='4', metadata={}, page_content='Embeddings are vector representations of text.')]

In [17]:
# Using 'similarity'

retrieval = vec_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 2} 
)

result = retrieval.invoke('What is langchain?')

result

# See it is returning both similar documents
# See the similarity score of both pairs in the below code

[Document(id='2', metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(id='1', metadata={}, page_content='LangChain makes it easy to work with LLMs.')]

In [18]:
# Checking similarity scores of both the pairs

from sklearn.metrics.pairwise import cosine_similarity

text1 = "LangChain is used to build LLM based applications."
text2 = "LangChain makes it easy to work with LLMs."

text3 = "LangChain is used to build LLM based applications."
text4 = "Embeddings are vector representations of text."

emb = OpenAIEmbeddings(
    model = 'text-embedding-3-large',
    dimensions = 32)

# Step 3: Embed texts
vec1 = emb.embed_query(text1)
vec2 = emb.embed_query(text2)

vec3 = emb.embed_query(text3)
vec4 = emb.embed_query(text4)

In [19]:
# Step 4: Compute cosine similarity
sim_1_2 = cosine_similarity([vec1], [vec2])
print("Cosine Similarity:", sim_1_2)

sim_3_4 = cosine_similarity([vec3], [vec4])
print("Cosine Similarity:", sim_3_4)


Cosine Similarity: [[0.88320549]]
Cosine Similarity: [[0.1866452]]


In [20]:
vec3

[-0.2927219867706299,
 0.0673079863190651,
 -0.28255802392959595,
 0.08436085283756256,
 -0.13100212812423706,
 -0.21333017945289612,
 -0.11609498411417007,
 0.11090007424354553,
 -0.1613810658454895,
 0.054461874067783356,
 0.07515682280063629,
 0.22857612371444702,
 -0.24664537608623505,
 -0.1890496164560318,
 0.07063950598239899,
 -0.005226674024015665,
 -0.3367657959461212,
 -0.1646561175584793,
 0.19176000356674194,
 0.04655657336115837,
 -0.002484522992745042,
 0.13755223155021667,
 -0.37493711709976196,
 0.12851759791374207,
 0.15155591070652008,
 -0.1700768917798996,
 0.1972937136888504,
 -0.11417512595653534,
 -0.22643038630485535,
 0.03732431307435036,
 0.17098034918308258,
 -0.06815498322248459]